# Part 2 of analysis


First we read the data from the `R` section. 

In [1]:
# Import packages required
import scipy.io, scipy.sparse as sp
import anndata as ad, pandas as pd

In [ ]:
m = scipy.io.mmread("cnmf_input/counts.mtx")
X = sp.csr_matrix(m).T.tocsr()
genes = [l.strip() for l in open("cnmf_input/genes.txt")]
cells = [l.strip() for l in open("cnmf_input/barcodes.txt")]
adata = ad.AnnData(X=X,
                   obs = pd.DataFrame(index = cells),
                   var = pd.DataFrame(index = genes),
                   )
adata.write("cnmf_input/counts.h5ad")

/tmp/ipykernel_45391/2948331793.py:1: DeprecationWarning: The default value for `spmatrix` is changing to `False` in v1.20.
             That means the default return type will be a sparse array.
             Unless you use * instead of @, ** for matrix power, or you depend
             on 2D shapes from e.g. `A.sum(axis=0)` it may not matter to you.
             See the spmatrix to sparray migration guide for details.
             https://docs.scipy.org/doc/scipy/reference/sparse.migration_to_sparray.html
             
  m = scipy.io.mmread("cnmf_input/counts.mtx")


We are going to use `cnmf` to identify specific programs in gene expressing. 

In [ ]:
cnmf prepare --output-dir ./cnmf_out --name mycNMF -c ~/SciP/cnmf_input/counts.h5ad -k 15 16 17 18 19 20 --n-iter 50 --seed 14 --numgenes 3000
cnmf factorize --output-dir ./cnmf_out --name mycNMF --worker-index 0 --total-workers 1

In [6]:
cnmf combine --output-dir ./cnmf_out --name mycNMF

SyntaxError: invalid syntax (1941028857.py, line 1)

In [ ]:
cnmf k_selection_plot --output-dir ./cnmf_out --name mycNMF 

In [ ]:
cnmf consensus --output-dir ./cnmf_out --name mycNMF --components 10 --local-density-threshold 0.01 --show-clustering

In [ ]:
from cnmf import cNMF
cnmf_obj = cNMF(output_dir="./cnmf_out", name="mycNMF") # This directory has to be relative 
# K and density_threshold must match the consensus run above (0.10 -> here 0.1)
usage, spectra_scores, spectra_tpm, top_genes = \
cnmf_obj.load_results(K=10, density_threshold=0.10)
usage = usage.div(usage.sum(axis=1), axis=0) # normalize per cell
usage.columns = [f"GEP{c}" for c in usage.columns] # tidy names
usage.to_csv("cnmf_out/usage_GEP.csv") # cells x programs
top_genes.head(100) # top genes per program

,1,2,3,4,5,6,7,8,9,10
0,MBNL1,TXNDC5,RPS18,PTPRB,PRAP1,HLA-DPB1,CALD1,HHIP,BMX,RIMS2
1,SKAP1,FNDC3B,RPL10,LIFR,APOA4,HLA-DRA,NOTCH3,NPNT,ITPRID1,PCSK1N
2,CD96,HERPUD1,RPL41,PTPRM,TMPRSS15,HLA-DRB1,DLC1,DES,IRAG2,RGS7
3,PTPRC,HSP90AA1,RPS2,NFIB,ALDOB,HLA-DQA1,PRKG1,FHL1,PSTPIP2,KCNH7
4,PRKCH,GNG7,RPL13,LDB2,FABP1,HLA-DQB1,ADRA1A,DMD,ENSG00000231698,CACNA1A
...,...,...,...,...,...,...,...,...,...,...
95,LCP1,STX5,TOMM7,CAVIN1,CYP2C9,SGK1,ANK2,TGFB1I1,HOXA3,PRLR
96,CD2,GLCCI1,COX7C,PTK2,PTGR1,TCOF1,TPM2,LAMB1,RUNX1,KIF1A
97,PTPRJ,GLA,SLC25A6,RNASE1,COBL,SAT1,PLN,BOC,CSMD1,UNC79
98,FNBP1,ENSG00000287979,ATP5F1E,ENSG00000259124,MEP1A,CYRIA,DAAM2,COL12A1,ENSG00000285695,CRYBA2


In [11]:
usage, spectra_scores, spectra_tpm, top_genes = cnmf_obj.load_results(K=12,density_threshold=0.10)

FileNotFoundError: [Errno 2] No such file or directory: './cnmf_out/mycNMF/mycNMF.gene_spectra_score.k_12.dt_0_1.txt'

In [ ]:
barrier_genes = ["TJP1", "OCLN", "CLDN1", "CLDN2", "CLDN3", "CLDN4", "CLDN7", "CDH1", "CTNNB1", "MUC2", "TFF3", "F11R"] 
genes_of_interest = barrier_genes + ["WNT5A"]

In [ ]:
available = [g for g in genes_of_interest if g in spectra_scores.index]
missing = [g for g in genes_of_interest if g not in spectra_scores.index]
print("Not found in matrix: ", missing)

Not found in matrix:  []


In [ ]:
gene_scores = spectra_scores.loc[available]
gene_scores.columns = [f'GEP{c}' for c in gene_scores.columns]
print(gene_scores)

            GEP1      GEP2      GEP3      GEP4      GEP5      GEP6      GEP7  \
TJP1   -0.000376 -0.000290 -0.000169  0.000954  0.000373 -0.000074  0.000313   
OCLN   -0.000154 -0.000113  0.000002  0.000067  0.000528 -0.000011 -0.000073   
CLDN1  -0.000058 -0.000039  0.000062 -0.000030  0.000047 -0.000008  0.000403   
CLDN2  -0.000041  0.000006  0.000005 -0.000019  0.000117 -0.000015 -0.000023   
CLDN3  -0.000330 -0.000096  0.000644 -0.000137  0.000600 -0.000136 -0.000144   
CLDN4  -0.000267 -0.000131  0.000321 -0.000118  0.000750 -0.000046 -0.000129   
CLDN7  -0.000268 -0.000205  0.000079 -0.000115  0.001200 -0.000067 -0.000132   
CDH1   -0.000253 -0.000114 -0.000202 -0.000117  0.001141 -0.000036 -0.000123   
CTNNB1  0.000114 -0.000205 -0.000724  0.000452  0.000156  0.000424 -0.000004   
MUC2   -0.000039 -0.000006  0.000068 -0.000013  0.000028 -0.000015 -0.000021   
TFF3   -0.000151 -0.000062  0.000441 -0.000012  0.000029 -0.000052 -0.000058   
F11R   -0.000171 -0.000083 -0.000179  0.

In [ ]:
gene_scores = spectra_tpm.loc[available]
gene_scores.columns = [f'GEP{c}' for c in gene_scores.columns]
print(gene_scores)

              GEP1       GEP2       GEP3        GEP4        GEP5        GEP6  \
TJP1      0.000000   0.000000   0.000000  619.593930  245.765950    0.000000   
OCLN      1.195318   0.754999   5.050063   29.536160   94.962930   11.180760   
CLDN1     0.000000   0.000000   0.415064    0.000000    2.297637    0.572308   
CLDN2     0.109724   1.142124   0.000000    0.000000    4.560452    0.088644   
CLDN3     0.000000   7.231420  32.082430    0.000000  147.904630    0.000000   
CLDN4     0.000000   1.753442  19.639074    0.000000  235.970400   18.894585   
CLDN7     0.000000   0.000000  34.607216    0.000000  488.503020    1.283402   
CDH1      0.000000  14.008441   0.000000    0.000000  490.969400   15.913312   
CTNNB1  203.009740  74.821450   0.000000  557.916300  290.525120  536.013100   
MUC2      0.000000   0.342410   2.897982    0.000000    8.322475    0.000000   
TFF3      0.000000   0.000000  89.635680   39.959126   59.448925    0.000000   
F11R      5.961338  13.471414   4.411547